# 3. Data Wrangling

**Objetivo:** limpiar el dataset de lanzamientos (`dataset_part_1.csv`, generado en el notebook
`01_data_collection_api`) y construir la variable objetivo (`Class`) que vamos a predecir con
Machine Learning: **¿la primera etapa del Falcon 9 aterrizó con éxito?**

**Flujo de trabajo:**

1. Cargar `dataset_part_1.csv`.
2. Calcular el porcentaje de valores faltantes por columna y revisar los tipos de datos.
3. Explorar cuántos lanzamientos hubo por sitio de lanzamiento y por órbita.
4. Analizar los distintos valores de `Outcome` (resultado del aterrizaje) y su frecuencia.
5. Etiquetar cada lanzamiento como éxito (`Class = 1`) o fracaso (`Class = 0`) según `Outcome`.
6. Calcular la tasa de éxito general.
7. Exportar el dataset limpio y etiquetado a `data/processed/dataset_part_2.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## Paso 1: cargar los datos

In [ ]:
df = pd.read_csv('../data/raw/dataset_part_1.csv')
df.head()

## Paso 2: valores faltantes y tipos de datos

In [ ]:
(df.isnull().sum() / len(df) * 100).round(2)

In [ ]:
df.dtypes

`LandingPad` va a tener nulos de forma esperada: son los lanzamientos donde no hubo
intento de aterrizaje en una plataforma (ASDS), sino en tierra (RTLS), en el océano, o
directamente no se intentó aterrizar. No lo tratamos como dato faltante a imputar.

## Paso 3: lanzamientos por sitio y por órbita

In [ ]:
df['LaunchSite'].value_counts()

In [ ]:
df['Orbit'].value_counts()

## Paso 4: resultados de aterrizaje (`Outcome`)

`Outcome` combina el resultado (`True`/`False`/`None`) con el tipo de aterrizaje intentado
(`ASDS` = drone ship, `RTLS` = vuelta al sitio de lanzamiento, `Ocean` = amerizaje, o `None`
si no se intentó aterrizar).

In [ ]:
landing_outcomes = df['Outcome'].value_counts()
landing_outcomes

## Paso 5: etiquetar éxito/fracaso (`Class`)

Un aterrizaje es **exitoso** si `Outcome` empieza con `"True"` (`True ASDS`, `True RTLS`,
`True Ocean`). Cualquier otro caso (`False ...`, o `None ...` cuando no hubo intento o no se
registró resultado) lo consideramos **fracaso** para el objetivo de predicción.

In [ ]:
bad_outcomes = set(landing_outcomes.keys()[~landing_outcomes.keys().str.startswith('True')])
bad_outcomes

In [ ]:
df['Class'] = df['Outcome'].apply(lambda outcome: 0 if outcome in bad_outcomes else 1)
df[['Outcome', 'Class']].drop_duplicates().sort_values('Outcome').reset_index(drop=True)

## Paso 6: tasa de éxito general

In [ ]:
success_rate = df['Class'].mean()
print(f"Tasa de aterrizajes exitosos: {success_rate:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
df['Class'].value_counts().sort_index().plot(kind='bar', ax=ax, color=['#c0392b', '#27ae60'])
ax.set_xticklabels(['Fracaso (0)', 'Éxito (1)'], rotation=0)
ax.set_ylabel('Cantidad de lanzamientos')
ax.set_title('Distribución de la variable objetivo (Class)')
plt.tight_layout()
plt.show()

## Paso 7: exportar el dataset limpio

In [ ]:
df.to_csv('../data/processed/dataset_part_2.csv', index=False)
df.shape

## Resumen

El dataset queda limpio, con una fila por lanzamiento y la columna `Class` (0 = fracaso,
1 = éxito) como variable objetivo. Con este archivo (`dataset_part_2.csv`) seguimos en el
notebook **04 - Análisis Predictivo (ML)** para entrenar y comparar modelos de clasificación.